# Experiment 9: Train-and-Test AWGN Ablation on the Waveform-Family Dataset

This notebook reruns the matched train/test AWGN ablation on the synthetic waveform-family dataset, then loads the saved results and plots the model accuracy curves.


In [3]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve().parent
SCRIPT = ROOT / 'experiments' / 'run_experiment9_awgn_train_test_waveform.py'
RESULTS_PATH = ROOT / 'experiments' / 'experiment9_awgn_train_test_waveform_results.json'
PLOT_PATH = ROOT / 'experiments' / 'experiment9_awgn_train_test_waveform.png'
PYTHON = ROOT / '.venv' / 'bin' / 'python'

print('root:', ROOT)
print('python:', PYTHON)


root: /home/klukasdh/Projects/DLSignalClassifier
python: /home/klukasdh/Projects/DLSignalClassifier/.venv/bin/python


In [4]:
cmd = [
    str(PYTHON),
    str(SCRIPT),
    '--epochs', '50',
    '--batch-size', '256',
    '--results-path', str(RESULTS_PATH),
    '--plot-path', str(PLOT_PATH),
]
completed = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, check=True)
print(completed.stdout)


KeyboardInterrupt: 

In [ ]:
results = json.loads(RESULTS_PATH.read_text())
rows = results['awgn_train_test_ablation']

snr = np.array([row['snr_db'] for row in rows], dtype=float)
iq = np.array([row['time_cnn']['test_acc'] for row in rows], dtype=float)
fft = np.array([row['fft_cnn']['test_acc'] for row in rows], dtype=float)
gated = np.array([row['gated_multimodal_cnn']['test_acc'] for row in rows], dtype=float)

print('dataset windows:', results['dataset']['train_windows'], results['dataset']['val_windows'], results['dataset']['test_windows'])
print('epochs:', results['training']['epochs'])
print('runtime_seconds:', round(results['runtime_seconds'], 2))

for row in rows:
    print(
        f"SNR {row['snr_db']:>5.1f} dB | "
        f"IQ {row['time_cnn']['test_acc']:.3f} | "
        f"FFT {row['fft_cnn']['test_acc']:.3f} | "
        f"Gated {row['gated_multimodal_cnn']['test_acc']:.3f}"
    )


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(snr, iq, marker='o', linewidth=2, label='IQ CNN')
ax.plot(snr, fft, marker='o', linewidth=2, label='FFT CNN')
ax.plot(snr, gated, marker='o', linewidth=2, label='Gated Multimodal CNN')
ax.set_xlabel('Train/Test AWGN SNR (dB)')
ax.set_ylabel('Test Accuracy')
ax.set_title('Experiment 9: Waveform-Family Train+Test AWGN Ablation')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
